# Phase 1 — Exploration des Données (EDA)

**Objectif :** Construire un dataset unifié, comprendre les données, identifier les problèmes de qualité.

**Sources :** 15 fichiers CSV journaliers dans `data/` — OHLCV multi-actifs + indicateurs BTC on-chain + sentiment.

**Cible :** `label_dir_1d` = 1 si `close_btc[t+1] > close_btc[t]`, sinon 0.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True

DATA_DIR = Path("data")

## 1. Chargement des sources

In [ ]:
def load_ohlcv(filename, suffix):
    """Charge un CSV OHLCV et renomme les colonnes avec le suffixe de l'actif."""
    df = pd.read_csv(DATA_DIR / filename, parse_dates=["Date"])
    df = df.rename(columns={
        "Date": "date",
        "Open":   f"open_{suffix}",
        "High":   f"high_{suffix}",
        "Low":    f"low_{suffix}",
        "Close":  f"close_{suffix}",
        "Volume": f"volume_{suffix}",
    })
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    return df.set_index("date")

def load_single(filename, col_name):
    """Charge un CSV Date,Value et renomme la colonne Value."""
    df = pd.read_csv(DATA_DIR / filename, parse_dates=["Date"])
    df = df.rename(columns={"Date": "date", "Value": col_name})
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    return df.set_index("date")

# --- OHLCV ---
btc    = load_ohlcv("btc_daily.csv",    "btc")
xau    = load_ohlcv("xau_daily.csv",    "gold")
eth    = load_ohlcv("eth_daily.csv",    "eth")
snp    = load_ohlcv("snp500_daily.csv", "sp500")
dxy    = load_ohlcv("dxy_daily.csv",    "dxy")
vix    = load_ohlcv("vix_daily.csv",    "vix")
us10y  = load_ohlcv("us10y_daily.csv",  "us10y")
oil    = load_ohlcv("oil_daily.csv",    "oil")
silver = load_ohlcv("silver_daily.csv", "silver")

# --- Date,Value ---
fedfunds    = load_single("fedfunds_daily.csv",         "fedfunds")
funding     = load_single("btc_funding_rate_daily.csv", "funding_rate")
hashrate    = load_single("btc_hashrate_daily.csv",     "hashrate")
mvrv        = load_single("btc_mvrv_daily.csv",         "mvrv")
nupl        = load_single("btc_nupl_daily.csv",         "nupl")
gtrends     = load_single("google_trends_bitcoin.csv",  "google_trends")

print("Sources chargées :")
for name, df in [("BTC", btc), ("Gold", xau), ("ETH", eth), ("S&P500", snp),
                 ("DXY", dxy), ("VIX", vix), ("US10Y", us10y), ("Oil", oil),
                 ("Silver", silver), ("FedFunds", fedfunds), ("Funding", funding),
                 ("Hashrate", hashrate), ("MVRV", mvrv), ("NUPL", nupl),
                 ("Google Trends", gtrends)]:
    print(f"  {name:<15} {len(df):>5} lignes   {df.index.min().date()} → {df.index.max().date()}")

## 2. Fusion sur la date (left join sur BTC)

In [ ]:
# Left join sur BTC : toutes les dates du calendrier BTC (7j/7) sont conservées.
# Les actifs à marchés fermés (week-ends, jours fériés) auront des NaN → forward-fill ensuite.
df = btc.copy()
for other in [xau, eth, snp, dxy, vix, us10y, oil, silver,
              fedfunds, funding, hashrate, mvrv, nupl, gtrends]:
    df = df.join(other, how="left")

df = df.sort_index()

print(f"Dataset fusionné : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"Période : {df.index.min().date()} → {df.index.max().date()}")
df.head(3)

## 3. Inspection générale

In [ ]:
print("=== Types ===")
print(df.dtypes.to_string())
print(f"\n=== Statistiques descriptives ===")
df.describe().T[["count", "mean", "std", "min", "max"]]

## 4. Création de la variable cible `label_dir_1d`

`label_dir_1d = 1` si le prix de clôture BTC du lendemain est supérieur à celui d'aujourd'hui, sinon `0`.

**Important :** la dernière ligne est supprimée (target inconnue) et la cible ne doit jamais apparaître comme feature.

In [ ]:
# shift(-1) : compare close[t] avec close[t+1]
df["label_dir_1d"] = (df["close_btc"].shift(-1) > df["close_btc"]).astype(int)

# Supprimer la dernière ligne : target inconnue (close[t+1] n'existe pas)
df = df.iloc[:-1]

print(f"Dataset après création de la cible : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"Période : {df.index.min().date()} → {df.index.max().date()}")
print(f"\nDistribution de label_dir_1d :")
print(df["label_dir_1d"].value_counts())
print(f"\nTaux de hausse : {df['label_dir_1d'].mean():.2%}")

## 5. Analyse des données manquantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
missing_df = missing_df[missing_df["missing"] > 0].sort_values("pct", ascending=False)

print(f"Colonnes avec valeurs manquantes ({len(missing_df)}/{df.shape[1]}) :\n")
print(missing_df.to_string())

# Visualisation
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(12, max(4, len(missing_df) * 0.4)))
    missing_df["pct"].plot(kind="barh", ax=ax, color="steelblue")
    ax.set_xlabel("% de valeurs manquantes")
    ax.set_title("Données manquantes par colonne")
    plt.tight_layout()
    plt.show()

## 6. Stratégie de gestion des NaN

- **Actifs marchés fermés** (gold, silver, oil, sp500, dxy, vix, us10y, fedfunds) : `forward-fill` — le dernier prix connu est propagé sur le week-end/jour férié.
- **ETH** : données depuis 2017 — NaN avant 2017 laissés tels quels (exclus du preprocessing via `dropna`).
- **Funding rate** : données depuis sept. 2019 — idem.
- **Google Trends** : mensuel → forward-fill vers quotidien.
- **Hashrate, MVRV, NUPL** : données quotidiennes depuis 2012, NaN résiduels rares → forward-fill.

In [ ]:
# Colonnes à forward-fill (marchés fermés + google trends mensuel + on-chain)
ffill_cols = [c for c in df.columns if any(c.endswith(s) for s in [
    "_gold", "_sp500", "_dxy", "_vix", "_us10y", "_oil", "_silver"
])] + ["fedfunds", "google_trends", "hashrate", "mvrv", "nupl"]

df[ffill_cols] = df[ffill_cols].ffill()

# Vérification post-ffill
missing_after = df.isnull().sum()
missing_after = missing_after[missing_after > 0]
print("NaN restants après forward-fill :")
print(missing_after.to_string() if len(missing_after) > 0 else "  Aucun (hors ETH pré-2017 et funding pré-2019)")

## 7. Détection des outliers (IQR)

In [ ]:
# Calcul sur les rendements journaliers (plus stables que les prix bruts)
close_cols = [c for c in df.columns if c.startswith("close_")]
returns = df[close_cols].pct_change().dropna()

outlier_summary = []
for col in close_cols:
    r = returns[col].dropna()
    q1, q3 = r.quantile(0.25), r.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 3 * iqr, q3 + 3 * iqr
    n_out = ((r < lower) | (r > upper)).sum()
    outlier_summary.append({
        "colonne": col,
        "n_outliers": n_out,
        "pct": round(n_out / len(r) * 100, 2),
        "min_return_%": round(r.min() * 100, 2),
        "max_return_%": round(r.max() * 100, 2),
    })

out_df = pd.DataFrame(outlier_summary).set_index("colonne")
print("Outliers sur les rendements journaliers (seuil 3×IQR) :")
print(out_df.to_string())
print("\nNote : Les outliers sont conservés — ils correspondent à des événements")
print("de marché réels (crashs, rallyes). RobustScaler en Phase 2 les atténuera.")

## 8. Visualisations

### 8.1 Prix BTC dans le temps

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Prix BTC
axes[0].plot(df.index, df["close_btc"], color="orange", linewidth=0.8)
axes[0].set_ylabel("Prix BTC (USD)")
axes[0].set_title("Prix de clôture Bitcoin (USD) — données journalières")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Colorer selon la cible
colors = df["label_dir_1d"].map({1: "green", 0: "red"})
axes[1].bar(df.index, df["close_btc"].pct_change() * 100,
            color=colors, width=1, alpha=0.6)
axes[1].set_ylabel("Rendement journalier (%)")
axes[1].set_title("Rendements journaliers (vert = hausse J+1, rouge = baisse J+1)")
axes[1].set_xlabel("Date")
axes[1].axhline(0, color="black", linewidth=0.5)

plt.tight_layout()
plt.show()

### 8.2 Distribution de la cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution globale
counts = df["label_dir_1d"].value_counts()
axes[0].bar(["Baisse (0)", "Hausse (1)"], counts.values,
            color=["tomato", "mediumseagreen"], edgecolor="white")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, f"{v}\n({v/len(df):.1%})", ha="center", fontsize=11)
axes[0].set_title("Distribution globale de label_dir_1d")
axes[0].set_ylabel("Nombre de jours")

# Distribution par année
df["year"] = df.index.year
yearly = df.groupby("year")["label_dir_1d"].mean()
axes[1].bar(yearly.index, yearly.values * 100, color="steelblue", edgecolor="white")
axes[1].axhline(50, color="red", linestyle="--", linewidth=1, label="50%")
axes[1].set_title("Taux de hausse J+1 par année (%)")
axes[1].set_ylabel("% jours haussiers")
axes[1].set_xlabel("Année")
axes[1].legend()
df.drop(columns="year", inplace=True)

plt.tight_layout()
plt.show()

### 8.3 Corrélations entre actifs (rendements journaliers)

In [ ]:
# Rendements log sur les prix de clôture (2019+ pour avoir tous les actifs)
close_cols = [c for c in df.columns if c.startswith("close_")]
log_ret = np.log(df[close_cols] / df[close_cols].shift(1)).dropna()
log_ret_recent = log_ret[log_ret.index >= "2019-01-01"]  # période commune à tous les actifs

# Noms courts pour l'affichage
short_names = {c: c.replace("close_", "") for c in close_cols}
corr = log_ret_recent.rename(columns=short_names).corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Corrélation des rendements journaliers log (2019–présent)")
plt.tight_layout()
plt.show()

### 8.4 Évolution des indicateurs on-chain et sentiment

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=False)
axes = axes.flatten()

indicators = [
    ("mvrv",         "MVRV Ratio",              "purple"),
    ("nupl",         "NUPL",                    "teal"),
    ("hashrate",     "Hash Rate (TH/s)",         "gray"),
    ("funding_rate", "Funding Rate perpétuel",   "darkorange"),
    ("google_trends","Google Trends 'bitcoin'",  "royalblue"),
    ("fedfunds",     "Fed Funds Rate (%)",        "firebrick"),
]

for ax, (col, title, color) in zip(axes, indicators):
    data = df[col].dropna()
    ax.plot(data.index, data.values, color=color, linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("")
    # Ligne horizontale de référence pour certains indicateurs
    if col == "mvrv":
        ax.axhline(1, color="black", linestyle="--", linewidth=0.7, alpha=0.5, label="MVRV=1")
        ax.legend(fontsize=8)
    if col == "nupl":
        ax.axhline(0, color="black", linestyle="--", linewidth=0.7, alpha=0.5)
    if col == "funding_rate":
        ax.axhline(0, color="black", linestyle="--", linewidth=0.7, alpha=0.5)

plt.suptitle("Indicateurs on-chain, macro et sentiment", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 9. Sauvegarde du dataset fusionné

Ce dataset est l'input de la Phase 2 (preprocessing). Il contient toutes les colonnes brutes avant feature engineering.

In [ ]:
out_path = DATA_DIR / "merged_daily.csv"
df.to_csv(out_path)
print(f"Dataset sauvegardé : {out_path}")
print(f"  Shape : {df.shape}")
print(f"  Colonnes : {list(df.columns)}")

## 10. Conclusions EDA

| Observation | Décision pour Phase 2 |
|---|---|
| BTC disponible depuis 2012, ETH depuis 2017, funding depuis sept. 2019 | Période ML effective : **sept. 2019 → présent** (toutes sources disponibles) |
| Actifs marchés fermés ont des NaN week-ends/fériés | **Forward-fill** appliqué |
| Google Trends : mensuel | **Forward-fill** vers daily déjà appliqué |
| Outliers sur rendements (>3×IQR) | **Conservés** — événements réels. RobustScaler les atténue |
| Cible proche de 50/50 (légère asymétrie haussière ~53%) | Pas de rééchantillonnage nécessaire, monitorer F1 par classe |
| Corrélations BTC/ETH élevées (~0.7+), BTC/SP500 modérée (~0.3–0.5) | Toutes les features gardées — le modèle sélectionnera |

**→ Prochaine étape : `02_preprocessing.py`** — feature engineering, split temporel, normalisation, export des splits.